<a href="https://colab.research.google.com/github/mendietasaul/clasificacion-Formalidad-Laboral-ENEMDU-2025/blob/main/Clasificador_informalidad_ENEMDU_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==============================================================================
# PASO 1: IMPORTACIÓN DE LIBRERÍAS Y CARGA DE DATOS
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Se aqjusta la ruta a la ubicación exacta de tu archivo en Drive o Colab
ruta_archivo = '/content/drive/MyDrive/UIDE/PROYECTO INGRESO/enemdu_2025_anual.csv'

# Se carga la base indicando delimitador y tratamiento de celdas vacías
df_raw = pd.read_csv(
    ruta_archivo,
    sep=';',
    encoding='utf-8-sig',
    na_values=[' ', ''],
    low_memory=False
)

# Estandarizar todos los nombres de columnas a minúsculas
df_raw.columns = df_raw.columns.str.lower()

# Ver cuántas filas y columnas tiene la base original completa
print(f"Dimensiones de la base completa: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas.")

Dimensiones de la base completa: 334786 filas y 139 columnas.


In [ ]:
# ==============================================================================
# PASO 2: FILTRADO DE POBLACIÓN Y SELECCIÓN DE VARIABLES
# ==============================================================================

# 1. Definir la lista de columnas que utilizaremos
columnas_modelo = [
    'secemp',   # Variable objetivo (Sector de empleo)
    'p02',      # Sexo
    'p03',      # Edad
    'area',     # Área urbana / rural
    'p06',      # Estado civil
    'nnivins',  # Nivel de instrucción (resumen oficial INEC)
    'p24',      # Horas trabajadas a la semana
    'p42',      # Categoría de ocupación
    'p47a',     # Tamaño del establecimiento
    'rama1',    # Rama de actividad económica
    'grupo1'    # Grupo ocupacional
]

# 2. Filtrar únicamente a la población ocupada objetivo:
#    - Edad >= 15 años
#    - secemp igual a 1 (Formal) o 2 (Informal)
filtro_ocupados = (df_raw['p03'] >= 15) & (df_raw['secemp'].isin(['1', '2', 1, 2]))

# Creamos una copia explícita con solo las columnas seleccionadas
df_trabajo = df_raw.loc[filtro_ocupados, columnas_modelo].copy()

# 3. Construir la variable dependiente binaria  del target_formal:
#    1 = Sector Formal
#    0 = Sector Informal
df_trabajo['secemp'] = df_trabajo['secemp'].astype(int)
df_trabajo['target_formal'] = (df_trabajo['secemp'] == 1).astype(int)

# Eliminamos la columna original 'secemp' para que el modelo no tenga fuga de datos
df_trabajo.drop(columns=['secemp'], inplace=True)

# 4. Inspección de resultados
print(f"Registros útiles para el modelo: {len(df_trabajo):,} filas.")
print(f"Número de variables predictoras: {df_trabajo.shape[1] - 1}")

print("\n--- Distribución del Target (Formalidad) ---")
conteo = df_trabajo['target_formal'].value_counts()
porcentaje = df_trabajo['target_formal'].value_counts(normalize=True) * 100
resumen_target = pd.DataFrame({'Total': conteo, 'Porcentaje (%)': porcentaje.round(2)})
resumen_target.index = ['Informal (0)', 'Formal (1)']
print(resumen_target)

Registros útiles para el modelo: 154,330 filas.
Número de variables predictoras: 10

--- Distribución del Target (Formalidad) ---
              Total  Porcentaje (%)
Informal (0)  83557           54.14
Formal (1)    70773           45.86


In [ ]:
# ==============================================================================
# PASO 3: DIAGNÓSTICO DE TIPOS Y VALORES NULOS
# ==============================================================================

# Asegurar conversión de variables numéricas continuas
df_trabajo['p03'] = pd.to_numeric(df_trabajo['p03'], errors='coerce')
df_trabajo['p24'] = pd.to_numeric(df_trabajo['p24'], errors='coerce')

# Diagnóstico general
resumen_diagnostico = pd.DataFrame({
    'Tipo': df_trabajo.dtypes,
    'Valores Nulos': df_trabajo.isnull().sum(),
    'Porcentaje Nulos (%)': (df_trabajo.isnull().sum() / len(df_trabajo) * 100).round(2)
})

print(resumen_diagnostico)

                  Tipo  Valores Nulos  Porcentaje Nulos (%)
p02              int64              0                   0.0
p03              int64              0                   0.0
area             int64              0                   0.0
p06            float64              0                   0.0
nnivins        float64              0                   0.0
p24            float64              0                   0.0
p42            float64              0                   0.0
p47a           float64              0                   0.0
rama1          float64              0                   0.0
grupo1         float64              0                   0.0
target_formal    int64              0                   0.0


In [ ]:
# ==============================================================================
# GUARDAR LA BASE FILTRADA Y LIMPIA EN DRIVE
# ==============================================================================
ruta_guardado = '/content/drive/MyDrive/UIDE/PROYECTO INGRESO/enemdu_2025_limpia.csv'

# Guardamos la base con separador punto y coma
df_trabajo.to_csv(ruta_guardado, index=False, sep=';', encoding='utf-8')

print("Base limpia y completa guardada con éxito en Google Drive.")

Base limpia y completa guardada con éxito en Google Drive.
